In [1]:
import pandas as pd


StatementMeta(, 07ed2710-4ae0-44e7-aea4-eedb86c7b34a, 3, Finished, Available, Finished, False)

In [9]:
# Correct path for Fabric Lakehouse Files
orders = spark.read.csv("Files/List of Orders.csv", header=True, inferSchema=True)

order_details = spark.read.csv("Files/Order Details.csv", header=True, inferSchema=True)

sales_target = spark.read.csv("Files/Sales target.csv", header=True, inferSchema=True)

print("Orders count:", orders.count())
print("Order Details count:", order_details.count())
print("Sales Target count:", sales_target.count())

orders.show(5)

StatementMeta(, 07ed2710-4ae0-44e7-aea4-eedb86c7b34a, 15, Finished, Available, Finished, False)

Orders count: 560
Order Details count: 1500
Sales Target count: 36
+--------+----------+------------+--------------+---------+
|Order ID|Order Date|CustomerName|         State|     City|
+--------+----------+------------+--------------+---------+
| B-25601|01-04-2018|      Bharat|       Gujarat|Ahmedabad|
| B-25602|01-04-2018|       Pearl|   Maharashtra|     Pune|
| B-25603|03-04-2018|       Jahan|Madhya Pradesh|   Bhopal|
| B-25604|03-04-2018|      Divsha|     Rajasthan|   Jaipur|
| B-25605|05-04-2018|     Kasheen|   West Bengal|  Kolkata|
+--------+----------+------------+--------------+---------+
only showing top 5 rows



In [10]:
from pyspark.sql.functions import col, to_date, dayofweek

orders_clean = orders.dropna().drop_duplicates().withColumnRenamed("Order ID","order_id").withColumnRenamed("Order Date", "order_date").withColumnRenamed("CustomerName","customer_name").withColumnRenamed("State","state").withColumnRenamed("City","city")

orders_clean = orders_clean.withColumn("order_date",to_date(col("order_date"),"dd-MM-yyyy"))

print("Clean Orders count:", orders_clean.count())
orders_clean.show(5)

StatementMeta(, 07ed2710-4ae0-44e7-aea4-eedb86c7b34a, 29, Finished, Available, Finished, False)

Clean Orders count: 500
+--------+----------+-------------+--------------+---------+
|order_id|order_date|customer_name|         state|     city|
+--------+----------+-------------+--------------+---------+
| B-25688|2018-06-11|       Swetha|Madhya Pradesh|   Indore|
| B-25786|2018-09-19|     Abhishek|     Karnataka|Bangalore|
| B-25839|2018-10-30|     Pranjali|   West Bengal|  Kolkata|
| B-25915|2018-12-19|      Sukruta|        Punjab| Amritsar|
| B-25943|2019-01-09|      Shardul|       Gujarat|Ahmedabad|
+--------+----------+-------------+--------------+---------+
only showing top 5 rows



In [11]:
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

print("Silver layer saved successfully!")

StatementMeta(, 07ed2710-4ae0-44e7-aea4-eedb86c7b34a, 31, Finished, Available, Finished, False)

Silver layer saved successfully!


In [16]:
from pyspark.sql.functions import sum, count, avg, round

# Read order details for joining
order_details_clean = order_details.dropna().dropDuplicates()

# Join orders with order details
df_joined = orders_clean.join(order_details_clean, 
                               orders_clean["order_id"] == order_details_clean["Order ID"], 
                               "inner")

# Gold layer — Sales by State
sales_by_state = df_joined.groupBy("state") \
                           .agg(sum("Profit").alias("total_profit"),
                                count("order_id").alias("total_orders"),
                                round(avg("Amount"),2).alias("avg_order_value"))

# Save Gold layer
sales_by_state.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_state")

print("Gold layer saved successfully!")
sales_by_state.show()

StatementMeta(, 07ed2710-4ae0-44e7-aea4-eedb86c7b34a, 62, Finished, Available, Finished, False)

Gold layer saved successfully!
+-----------------+------------+------------+---------------+
|            state|total_profit|total_orders|avg_order_value|
+-----------------+------------+------------+---------------+
|         Nagaland|       148.0|          45|         264.51|
|        Karnataka|       645.0|          54|         278.85|
|       Tamil Nadu|     -2216.0|          25|         243.48|
|   Andhra Pradesh|      -496.0|          42|         315.62|
|   Madhya Pradesh|      5551.0|         340|         309.24|
|           Punjab|      -609.0|          60|         279.77|
|              Goa|       370.0|          43|         155.93|
| Himachal Pradesh|       656.0|          29|         298.83|
|Jammu and Kashmir|         8.0|          49|          221.0|
|          Haryana|      1325.0|          26|         340.88|
|          Gujarat|       465.0|          87|         242.05|
|           Sikkim|       401.0|          24|         219.83|
|            Delhi|      2987.0|       